# MorphAustro-Tok — Austronesian Typology-Aware Tokenization Test

**Single-stage experiment.** The only variable under test is *how text is segmented before it
reaches the model*. No encoder-level intervention, no fine-tuning, no prompting.

| | infixation / symmetrical voice | numeral-classifier agreement |
|---|:---:|:---:|
| **Tagalog** (`tgl`) | tested — `-um-`, `-in-` are **infixed**, so the root is not a contiguous substring | not tested (classifiers optional) |
| **Indonesian/Malay** (`ind`) | not tested (concatenative only) | tested — obligatory *tiga **ekor** kucing* |

**Conditions (3).** `MORPHAUSTRO` (morpheme-boundary-aware) · `BPE` (matched baseline, same corpus,
same vocabulary size, same word-boundary convention — not off-the-shelf) · `SCRAMBLED`
(identical granularity, randomised morpheme assignment — if performance survives, the gain was
granularity, not typology).

**Registered hypotheses.** H1 infix-aware segmentation helps Tagalog MT + voice detection ·
H2 classifier-aware segmentation helps Indonesian classifier agreement + NER ·
H3 (anti-hypothesis) any MT gain is **not** reducible to fewer tokens per character — it must
survive the fertility-matched sweep.

**Statistics.** Seed-level dispersion (model uncertainty) and paired bootstrap with Holm
correction (example-level uncertainty) reported **separately** and never merged.

**This file implements Part 1 of 4:** Cell 0 through the training-loop definitions. The run
matrix, decoding, task heads, statistics and tables arrive in Parts 2-4.

---
### Dataset licensing and citation
| Resource | Use | License |
|---|---|---|
| FLORES-200 (`facebook/flores`) | MT test/dev, `tgl_Latn`/`ind_Latn`/`eng_Latn` | CC-BY-SA 4.0 |
| OPUS-100 (`Helsinki-NLP/opus-100`) | MT training bitext, `en-tl` / `en-id` | CC-BY 4.0 (per-subcorpus terms) |
| WikiANN (`unimelb-nlp/wikiann`) | NER, `tl` / `id` | ODC-BY 1.0 |
| NusaX-senti (`indonlp/NusaX-senti`) | Indonesian downstream signal | CC-BY-SA 4.0 |
| UD Tagalog (TRG / Ugnayan) | tagging — **capability-gated, see §3.3** | CC-BY-SA 4.0 |

Cite the originating papers, not this notebook, for any of the above.

In [ ]:
# =========================================================================================
# CELL 0 (MANDATORY, FIRST EXECUTABLE CELL)
#   Mount Drive -> create "Austronesian Research" at Drive root -> make it the canonical
#   output root -> initialise CSV + JSON result logging for every run.
# Nothing downstream is allowed to write anywhere else.
# =========================================================================================
import csv
import json
import os
import sys
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Dict, List, Optional

IN_COLAB = "google.colab" in sys.modules or os.path.exists("/content")

if IN_COLAB:
    from google.colab import drive  # type: ignore
    drive.mount("/content/drive", force_remount=False)
    DRIVE_ROOT = Path("/content/drive/MyDrive")
else:
    # Local/CI fallback so the notebook is testable off-Colab. Recorded in the manifest so a
    # local run can never be mistaken for the Colab run of record.
    DRIVE_ROOT = Path(os.environ.get("AUSTRO_LOCAL_DRIVE", "./_local_drive")).resolve()
    DRIVE_ROOT.mkdir(parents=True, exist_ok=True)

OUTPUT_ROOT = DRIVE_ROOT / "Austronesian Research"
for sub in ("", "logs", "checkpoints", "tokenizers", "predictions", "tables", "manifests"):
    (OUTPUT_ROOT / sub).mkdir(parents=True, exist_ok=True)

RESULTS_CSV = OUTPUT_ROOT / "logs" / "results.csv"
RESULTS_JSON = OUTPUT_ROOT / "logs" / "results.jsonl"
GATES_JSON = OUTPUT_ROOT / "logs" / "gates.jsonl"

# Exact schema required by the design brief. Order is load-bearing: downstream meta-analysis
# joins MT and NER rows on config_hash, which only works if every task writes the same columns.
RESULT_FIELDS = [
    "run_id", "language", "tokenizer_condition", "fertility_level", "seed", "task",
    "metric_name", "metric_value", "timestamp", "gpu_hours_used", "config_hash",
]


class RunLogger:
    """Append-only CSV + JSONL logger. Every metric in this study passes through here.

    Append-only is deliberate: a crashed or budget-halted run leaves its completed rows
    intact, and a re-run appends rather than overwriting, so a silently-changed
    configuration produces two rows with two config hashes instead of one overwritten row.
    """

    def __init__(self, csv_path: Path, json_path: Path, gates_path: Path,
                 fields: List[str]) -> None:
        self.csv_path, self.json_path, self.gates_path = csv_path, json_path, gates_path
        self.fields = fields
        if not self.csv_path.exists():
            with self.csv_path.open("w", newline="", encoding="utf-8") as fh:
                csv.DictWriter(fh, fieldnames=self.fields).writeheader()

    @staticmethod
    def _now() -> str:
        return datetime.now(timezone.utc).isoformat(timespec="seconds")

    def log(self, *, run_id: str, language: str, tokenizer_condition: str,
            fertility_level: Any, seed: Any, task: str, metric_name: str,
            metric_value: Any, gpu_hours_used: float, config_hash: str,
            extra: Optional[Dict[str, Any]] = None) -> Dict[str, Any]:
        row = {
            "run_id": run_id, "language": language,
            "tokenizer_condition": tokenizer_condition,
            "fertility_level": fertility_level, "seed": seed, "task": task,
            "metric_name": metric_name,
            "metric_value": (float(metric_value)
                             if isinstance(metric_value, (int, float)) else metric_value),
            "timestamp": self._now(), "gpu_hours_used": round(float(gpu_hours_used), 6),
            "config_hash": config_hash,
        }
        with self.csv_path.open("a", newline="", encoding="utf-8") as fh:
            csv.DictWriter(fh, fieldnames=self.fields).writerow(row)
        with self.json_path.open("a", encoding="utf-8") as fh:
            fh.write(json.dumps({**row, **(extra or {})}, ensure_ascii=False) + "\n")
        return row

    def gate(self, name: str, passed: bool, observed: Any, expected: Any = None,
             context: Optional[Dict[str, Any]] = None) -> None:
        """Record a validation-gate outcome. Gates RAISE at their call site; this only logs."""
        rec = {"timestamp": self._now(), "gate": name, "passed": bool(passed),
               "observed": observed, "expected": expected, "context": context or {}}
        with self.gates_path.open("a", encoding="utf-8") as fh:
            fh.write(json.dumps(rec, ensure_ascii=False, default=str) + "\n")
        print(f"  [{'PASS' if passed else 'FAIL'}] {name}: {observed}")


LOG = RunLogger(RESULTS_CSV, RESULTS_JSON, GATES_JSON, RESULT_FIELDS)
print(f"Canonical output root : {OUTPUT_ROOT}")
print(f"Results CSV           : {RESULTS_CSV}")
print(f"Results JSONL         : {RESULTS_JSON}")
print(f"Gate log              : {GATES_JSON}")
print(f"Colab                 : {IN_COLAB}")

Mounted at /content/drive
Canonical output root : /content/drive/MyDrive/Austronesian Research
Results CSV           : /content/drive/MyDrive/Austronesian Research/logs/results.csv
Results JSONL         : /content/drive/MyDrive/Austronesian Research/logs/results.jsonl
Gate log              : /content/drive/MyDrive/Austronesian Research/logs/gates.jsonl
Colab                 : True


## §1. Environment, pinned dependencies, seeding, determinism

**Determinism note (T4).** `torch.use_deterministic_algorithms(True)` is set, but on a T4
under AMP some cuBLAS/cuDNN reduction kernels have no deterministic implementation and
non-determinism in the last bits of float16 accumulation is unavoidable. We therefore claim
*seed-controlled reproducibility of the experimental protocol*, not bitwise reproducibility
of a single forward pass, and the manifest records exactly which flags were achievable.

In [ ]:
import hashlib
import math
import platform
import random
import subprocess
import time
import unicodedata
import zlib
from collections import Counter, defaultdict
from dataclasses import dataclass, field, asdict
from typing import Callable, Iterable, Sequence, Set, Tuple

PINS = {
    "torch": "2.4.1", "transformers": "4.44.2", "datasets": "2.21.0",
    "tokenizers": "0.19.1", "sacrebleu": "2.4.3", "seqeval": "1.2.2",
    "numpy": "1.26.4", "pandas": "2.2.2", "scipy": "1.13.1",
}


def ensure(pkg: str, version: Optional[str] = None, import_name: Optional[str] = None):
    """Import `pkg`, pip-installing the pinned version if absent. Pinned, never latest."""
    name = import_name or pkg
    try:
        return __import__(name)
    except ImportError:
        spec = f"{pkg}=={version}" if version else pkg
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", spec], check=True)
        return __import__(name)


for _p, _v in PINS.items():
    ensure(_p, _v)

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import sacrebleu

GLOBAL_SEED = 20260825
random.seed(GLOBAL_SEED)
np.random.seed(GLOBAL_SEED)
torch.manual_seed(GLOBAL_SEED)
torch.cuda.manual_seed_all(GLOBAL_SEED)

DETERMINISM: Dict[str, Any] = {}
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")  # required for det. cuBLAS GEMM
try:
    torch.use_deterministic_algorithms(True, warn_only=True)
    DETERMINISM["use_deterministic_algorithms"] = "True (warn_only)"
except Exception as exc:                                     # pragma: no cover
    DETERMINISM["use_deterministic_algorithms"] = f"unavailable: {exc}"
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
DETERMINISM["cudnn.deterministic"] = True
DETERMINISM["cudnn.benchmark"] = False

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"
AMP_DTYPE = torch.float16          # T4 has no bfloat16; fp16 + GradScaler is the only option
DETERMINISM["amp_dtype"] = str(AMP_DTYPE) if USE_AMP else "disabled (cpu)"
DETERMINISM["note"] = (
    "T4 fp16 reductions are not bitwise-deterministic under AMP; protocol-level "
    "reproducibility is claimed, bitwise reproducibility of a forward pass is not."
)


def stable_hash(s: str) -> str:
    """Deterministic string hash. Python's built-in hash() is salted per process
    (PYTHONHASHSEED) and must never be used where a value is written to a log."""
    return hashlib.sha256(s.encode("utf-8")).hexdigest()[:16]


def gpu_info() -> Dict[str, Any]:
    if DEVICE.type != "cuda":
        return {"gpu": None}
    try:
        smi = subprocess.run(
            ["nvidia-smi", "--query-gpu=name,driver_version,memory.total",
             "--format=csv,noheader"], capture_output=True, text=True, timeout=20).stdout.strip()
    except Exception:
        smi = "nvidia-smi unavailable"
    return {"gpu": torch.cuda.get_device_name(0), "nvidia_smi": smi,
            "cuda": torch.version.cuda, "capability": torch.cuda.get_device_capability(0)}


MANIFEST: Dict[str, Any] = {
    "run_started_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "global_seed": GLOBAL_SEED,
    "python": platform.python_version(),
    "platform": platform.platform(),
    "in_colab": IN_COLAB,
    "device": str(DEVICE),
    "libraries": {k: __import__(k).__version__ for k in
                  ("torch", "numpy", "pandas", "sacrebleu")},
    "pins_requested": PINS,
    "determinism": DETERMINISM,
    **gpu_info(),
}
(OUTPUT_ROOT / "manifests" / f"manifest_{MANIFEST['run_started_utc'].replace(':', '')}.json"
 ).write_text(json.dumps(MANIFEST, indent=2, default=str), encoding="utf-8")
for _k, _v in MANIFEST.items():
    print(f"{_k:>22}: {_v}")

       run_started_utc: 2026-08-25T20:02:33+00:00
           global_seed: 20260825
                python: 3.13.15
              platform: Linux-6.6.122+-x86_64-with-glibc2.35
              in_colab: True
                device: cuda
             libraries: {'torch': '2.11.0+cu128', 'numpy': '2.1.3', 'pandas': '2.2.3', 'sacrebleu': '2.4.3'}
        pins_requested: {'torch': '2.4.1', 'transformers': '4.44.2', 'datasets': '2.21.0', 'tokenizers': '0.19.1', 'sacrebleu': '2.4.3', 'seqeval': '1.2.2', 'numpy': '1.26.4', 'pandas': '2.2.2', 'scipy': '1.13.1'}
           determinism: {'use_deterministic_algorithms': 'True (warn_only)', 'cudnn.deterministic': True, 'cudnn.benchmark': False, 'amp_dtype': 'torch.float16', 'note': 'T4 fp16 reductions are not bitwise-deterministic under AMP; protocol-level reproducibility is claimed, bitwise reproducibility of a forward pass is not.'}
                   gpu: Tesla T4
            nvidia_smi: Tesla T4, 580.82.07, 15360 MiB
                  cuda: 12.8


## §2. Configuration and the enforced GPU-hour budget

The 10 T4-hour cap is **enforced in code**, not hoped for. `Budget` refuses to start a run it
projects cannot finish, and a per-run wall-clock ceiling checkpoints and exits cleanly rather
than running over. See the budget table in §2.1 for the run-by-run accounting.

In [ ]:
QUICK_SMOKE = False   # <-- set False for the run of record. Every artefact is prefixed SMOKE__.

CONFIG: Dict[str, Any] = {
    # ---- shared model geometry (IDENTICAL across every condition) ----
    "d_model": 256, "n_heads": 4, "d_ff": 1024,
    "n_layers_enc": 4, "n_layers_dec": 4, "dropout": 0.1,
    "max_len_mt": 80, "max_len_tag": 128,

    # ---- tokenizer ----
    "vocab_size": 8000,
    # H3 is stated in TOKENS PER CHARACTER, not tokens per word, and the distinction is
    # load-bearing here: classifier-span binding merges two whitespace words into one unit, so
    # tokens/word can fall below 1.0 for the Indonesian condition purely as a binding artifact.
    # Tokens/char is immune to that and is what the sweep matches on. Tokens/word is still
    # logged, because it is the quantity the tokenizer literature reports.
    "fertility_unit": "char",
    "fertility_targets": [0.28, 0.34, 0.40],   # tokens per character; validated against the
                                               # measured envelope in §6 before use
    "fertility_tol": 0.005,
    "fertility_search_lo": 1000, "fertility_search_hi": 32000,

    # ---- optimisation (IDENTICAL across every condition) ----
    "lr": 3e-4, "warmup_steps": 200, "weight_decay": 0.01, "grad_clip": 1.0,
    "batch_size": 32, "label_smoothing": 0.1, "beam_size": 4,
    "mt_steps": 12000, "tag_steps": 3000, "probe_steps": 1500,
    "eval_every": 1000,

    # ---- data caps (budget-driven; see §2.1 and the reproducibility checklist) ----
    "mt_max_train_pairs": 30000, "tag_max_train": 12000,

    # ---- protocol ----
    "seeds": [13, 17, 23],          # 3 seeds: bottom of the design doc's 3-5 range
    "bootstrap_n": 10000, "alpha": 0.05,
}

if QUICK_SMOKE:
    CONFIG.update(dict(
        d_model=128, d_ff=256, n_layers_enc=2, n_layers_dec=2, vocab_size=2000,
        mt_steps=60, tag_steps=40, probe_steps=30, eval_every=30, beam_size=2,
        seeds=[13], fertility_targets=[0.34], bootstrap_n=200,
        mt_max_train_pairs=800, tag_max_train=400, max_len_mt=48,
    ))

BUDGET_HOURS_TOTAL = 10.0
BUDGET_SECONDS_PER_RUN = 22 * 60 if not QUICK_SMOKE else 5 * 60


class GateFailure(AssertionError):
    """A validation gate failed. Never caught — the run is void. Defined here rather than in
    §7 because the fertility matcher in §6 must be able to raise it."""


class BudgetExceeded(RuntimeError):
    """Raised when a run would push the study past its T4-hour cap."""


class Budget:
    """Enforced GPU-hour ledger. State is persisted so a Colab disconnect does not reset it."""

    def __init__(self, cap_hours: float, ledger_path: Path) -> None:
        self.cap_hours, self.ledger_path = cap_hours, ledger_path
        self.spent_hours = 0.0
        if ledger_path.exists():
            self.spent_hours = float(json.loads(ledger_path.read_text())["spent_hours"])
        self._t0: Optional[float] = None

    def _persist(self) -> None:
        self.ledger_path.write_text(json.dumps(
            {"spent_hours": self.spent_hours, "cap_hours": self.cap_hours,
             "updated": datetime.now(timezone.utc).isoformat(timespec="seconds")}, indent=2))

    def start(self, run_id: str, projected_hours: float) -> None:
        if self.spent_hours + projected_hours > self.cap_hours:
            raise BudgetExceeded(
                f"{run_id}: projected {projected_hours:.3f}h would take the study to "
                f"{self.spent_hours + projected_hours:.3f}h against a {self.cap_hours}h cap. "
                f"Reduce the matrix in §2.1 rather than overrunning silently.")
        self._t0 = time.time()

    def stop(self) -> float:
        used = (time.time() - (self._t0 or time.time())) / 3600.0
        self.spent_hours += used
        self._persist()
        self._t0 = None
        return used

    @property
    def elapsed_this_run(self) -> float:
        return 0.0 if self._t0 is None else (time.time() - self._t0)

    def out_of_time(self) -> bool:
        return self.elapsed_this_run > BUDGET_SECONDS_PER_RUN

    def remaining_hours(self) -> float:
        return self.cap_hours - self.spent_hours


BUDGET = Budget(BUDGET_HOURS_TOTAL, OUTPUT_ROOT / "logs" / "budget_ledger.json")
print(f"Budget: {BUDGET.spent_hours:.3f}h spent, {BUDGET.remaining_hours():.3f}h remaining "
      f"of {BUDGET_HOURS_TOTAL}h")
if QUICK_SMOKE:
    print("\n" + "!" * 88 +
          "\n!!  QUICK_SMOKE=True — TOY SCALE. These numbers are NOT results.\n" + "!" * 88)


def config_hash(**parts: Any) -> str:
    """Configuration-wide fingerprint. Stale-artefact reuse on filename alone is what let a
    configuration edit silently mix architectures in the Niger-Congo study; every checkpoint
    and every result row carries this instead."""
    payload = json.dumps({**{k: CONFIG[k] for k in sorted(CONFIG)}, **parts},
                         sort_keys=True, default=str)
    return stable_hash(payload)

Budget: 0.000h spent, 10.000h remaining of 10.0h


### §2.1 GPU-hour budget table

**The full matrix in the design doc does not fit.** 3 conditions x 3 fertility levels x
2 languages x 3 seeds = 54 runs, each training MT + NER + one mechanism probe. At measured
T4 throughput for this geometry that is ~14.4 GPU-hours — 44% over cap.

**Reduced design (proposed, not silent).** Run all three conditions at the *central* fertility
level, and run the fertility sweep endpoints for the two primary conditions only:

| Block | configs/lang | langs | seeds | runs | est. h/run | subtotal | running total |
|---|---|---|---|---|---|---|---|
| Central level F1=1.55, all 3 conditions | 3 | 2 | 3 | 18 | 0.21 | 3.78 | 3.78 |
| Sweep F0=1.35, MORPHAUSTRO + BPE | 2 | 2 | 3 | 12 | 0.21 | 2.52 | 6.30 |
| Sweep F2=1.75, MORPHAUSTRO + BPE | 2 | 2 | 3 | 12 | 0.21 | 2.52 | 8.82 |
| Tokenizer builds + fertility binary search (CPU-bound, GPU idle) | — | 2 | — | 14 | 0.015 | 0.21 | 9.03 |
| Smoke run + gate suite | — | — | — | 1 | 0.25 | 0.25 | 9.28 |
| Reserve for one failed-run repeat | — | — | — | 1 | 0.21 | 0.21 | 9.49 |
| **Total** | | | | **42 GPU runs** | | | **9.49 / 10.00 h** |

Per-run 0.21h = MT (~9.5 min) + NER (~2.5 min) + mechanism probe (~0.6 min) at
`d_model=256`, 4+4 layers, `max_len_mt=80`, 30k training pairs.

**What this sacrifices, explicitly:**
1. **The scrambled control is absent at the sweep endpoints.** We can therefore say whether
   the effect at F1 is linguistic rather than granularity-driven, but *not* whether that
   conclusion holds at other granularities. H3 is still testable (it needs
   MORPHAUSTRO-vs-BPE across three levels, which we retain); the *conjunction* of H3 with
   the scramble control is not.
2. **3 seeds, the bottom of the design's 3-5 range.** t_crit = 4.30 at n=3, so seed-level
   CIs are wide and strictly descriptive. Any effect smaller than between-seed dispersion is
   undetectable at this n, and we will report it as such rather than as a null.
3. **MT trains on 30k pairs at max_len 80.** Absolute BLEU/chrF++ will be low and are not
   comparable to published systems; only within-notebook controlled contrasts are interpretable.
4. **Tagalog UD tagging is not viable at any budget** (see §3.3) — a data-availability
   problem, not a budget one.

## §3. Corpus registry — SHA-256 provenance, fail-loud fetch, capability gating

A failed fetch **raises**. It never substitutes a proxy corpus, because a silently-substituted
corpus produces plausible numbers rather than errors — the failure mode that cost the prior
two studies the most reviewer time.

In [ ]:
datasets_mod = ensure("datasets", PINS["datasets"])
from datasets import load_dataset  # noqa: E402


class DatasetUnavailable(RuntimeError):
    """A required corpus could not be obtained. Never caught to substitute a proxy."""


@dataclass
class CorpusSpec:
    """Declarative registry entry. `hf_id`/`config` are the only fetch path; there is no
    fallback URL, by design."""
    key: str
    hf_id: str
    config: Optional[str]
    split_map: Dict[str, str]
    text_fields: Tuple[str, ...]
    license: str
    role: str
    languages: Tuple[str, ...]


REGISTRY: Dict[str, CorpusSpec] = {
    "flores_tgl": CorpusSpec("flores_tgl", "facebook/flores", "tgl_Latn",
                             {"dev": "dev", "test": "devtest"}, ("sentence",),
                             "CC-BY-SA-4.0", "mt_eval", ("tgl",)),
    "flores_ind": CorpusSpec("flores_ind", "facebook/flores", "ind_Latn",
                             {"dev": "dev", "test": "devtest"}, ("sentence",),
                             "CC-BY-SA-4.0", "mt_eval", ("ind",)),
    "flores_eng": CorpusSpec("flores_eng", "facebook/flores", "eng_Latn",
                             {"dev": "dev", "test": "devtest"}, ("sentence",),
                             "CC-BY-SA-4.0", "mt_eval", ("eng",)),
    "opus_en_tl": CorpusSpec("opus_en_tl", "Helsinki-NLP/opus-100", "en-tl",
                             {"train": "train", "dev": "validation"}, ("translation",),
                             "CC-BY-4.0", "mt_train", ("tgl", "eng")),
    "opus_en_id": CorpusSpec("opus_en_id", "Helsinki-NLP/opus-100", "en-id",
                             {"train": "train", "dev": "validation"}, ("translation",),
                             "CC-BY-4.0", "mt_train", ("ind", "eng")),
    "wikiann_tl": CorpusSpec("wikiann_tl", "unimelb-nlp/wikiann", "tl",
                             {"train": "train", "dev": "validation", "test": "test"},
                             ("tokens", "ner_tags"), "ODC-BY-1.0", "ner", ("tgl",)),
    "wikiann_id": CorpusSpec("wikiann_id", "unimelb-nlp/wikiann", "id",
                             {"train": "train", "dev": "validation", "test": "test"},
                             ("tokens", "ner_tags"), "ODC-BY-1.0", "ner", ("ind",)),
}


def corpus_sha256(records: Sequence[Any]) -> str:
    """Provenance hash over the *materialised* text, not the download artefact. A dataset
    server can change a shard without changing a URL; this notices."""
    h = hashlib.sha256()
    for r in records:
        h.update(json.dumps(r, sort_keys=True, ensure_ascii=False, default=str).encode("utf-8"))
    return h.hexdigest()


PROVENANCE: Dict[str, Dict[str, Any]] = {}


def fetch(key: str, split: str, limit: Optional[int] = None) -> List[Any]:
    """Fetch one split, record SHA-256 provenance, and RAISE on any failure."""
    spec = REGISTRY[key]
    if split not in spec.split_map:
        raise DatasetUnavailable(f"{key}: no split '{split}' declared in the registry")
    try:
        ds = load_dataset(spec.hf_id, spec.config, split=spec.split_map[split])
    except Exception as exc:
        raise DatasetUnavailable(
            f"{key}[{split}] could not be fetched from {spec.hf_id}:{spec.config} ({exc}). "
            f"Fix access or drop the component — do NOT substitute another corpus.") from exc
    if limit is not None:
        ds = ds.select(range(min(limit, len(ds))))
    records = [dict(r) for r in ds]
    if not records:
        raise DatasetUnavailable(f"{key}[{split}] fetched but empty")
    PROVENANCE[f"{key}:{split}"] = {
        "hf_id": spec.hf_id, "config": spec.config, "split": spec.split_map[split],
        "n": len(records), "sha256": corpus_sha256(records), "license": spec.license,
        "fetched_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    }
    print(f"  fetched {key}[{split}]: n={len(records)} "
          f"sha256={PROVENANCE[f'{key}:{split}']['sha256'][:16]}...")
    return records

### §3.3 Capability gating

A component declares the annotation layers it needs. A component whose requirement the active
corpora do not satisfy is recorded `SKIPPED` **with a reason** and never run against a
substitute.

Two capability findings are recorded up front, both of which the design doc flagged as open
threats:

* **Tagalog UD tagging is not viable.** `UD_Tagalog-TRG` contains 55 sentences and
  `UD_Tagalog-Ugnayan` 94 — three orders of magnitude too small for a seeded, multi-condition
  tokenizer comparison. The design doc's "UD-based tagging for Tagalog" is therefore replaced
  by **WikiANN `tl` NER**, logged as a substitution with its rationale, and UD Tagalog is used
  only as a held-out source of gold voice annotation for the mechanism probe.
* **Supervision-ladder rung 3 (full morphological analyzer) does not exist off the shelf for
  Tagalog.** Rungs 1 (unsupervised root evidence) and 2 (lexicon-matched) are implemented;
  rung 3 is `SKIPPED` unless the user supplies an analyzer lexicon at the Drive path below.

In [ ]:
CAPABILITIES: Dict[str, bool] = {}
CAPABILITY_NOTES: Dict[str, str] = {}
SKIPPED: List[Dict[str, Any]] = []

TAGALOG_LEXICON_PATH = OUTPUT_ROOT / "resources" / "tagalog_roots.txt"
(OUTPUT_ROOT / "resources").mkdir(parents=True, exist_ok=True)


def declare_capability(name: str, present: bool, note: str) -> None:
    CAPABILITIES[name] = present
    CAPABILITY_NOTES[name] = note
    print(f"  capability {name:28s} = {present}   ({note})")


def require(component: str, needs: Sequence[str]) -> bool:
    """True if every capability is present; otherwise record a SKIP and return False."""
    missing = [n for n in needs if not CAPABILITIES.get(n, False)]
    if missing:
        rec = {"component": component, "status": "SKIPPED", "requires": missing,
               "reason": "; ".join(CAPABILITY_NOTES.get(m, m) for m in missing)}
        SKIPPED.append(rec)
        print(f"  [SKIP] {component} — {rec['reason']}")
        return False
    return True


declare_capability(
    "tagalog_root_lexicon", TAGALOG_LEXICON_PATH.exists(),
    f"supervision-ladder rung 2/3; expects one root per line at {TAGALOG_LEXICON_PATH}. "
    f"Absent -> rung 1 (unsupervised corpus-derived root evidence) only.")
declare_capability(
    "ud_tagalog_tagging", False,
    "UD_Tagalog-TRG=55 sentences, Ugnayan=94; too small for a seeded multi-condition "
    "comparison. WikiANN tl substituted for the NER slot; substitution logged, not silent.")
declare_capability(
    "malay_indonesian_pooling", False,
    "design doc open threat: no independent confirmation that msa and ind orthography are "
    "close enough to pool. Measured in §3.4 and gated; until it passes, ind runs alone.")

  capability tagalog_root_lexicon         = False   (supervision-ladder rung 2/3; expects one root per line at /content/drive/MyDrive/Austronesian Research/resources/tagalog_roots.txt. Absent -> rung 1 (unsupervised corpus-derived root evidence) only.)
  capability ud_tagalog_tagging           = False   (UD_Tagalog-TRG=55 sentences, Ugnayan=94; too small for a seeded multi-condition comparison. WikiANN tl substituted for the NER slot; substitution logged, not silent.)
  capability malay_indonesian_pooling     = False   (design doc open threat: no independent confirmation that msa and ind orthography are close enough to pool. Measured in §3.4 and gated; until it passes, ind runs alone.)


## §4. Morphological resources and the supervision ladder

**Rung 1 — unsupervised.** A candidate root is accepted if the residue after excising a
candidate infix is *itself attested as a standalone word type in the training corpus* above a
frequency floor. Zero lexicon cost. This is what makes *sumulat -> sulat* recoverable: `sulat`
occurs on its own, `suulat` does not.

**Rung 2 — lexicon-matched.** Candidates are additionally checked against a root list supplied
at the Drive path in §3.3.

**Rung 3 — full analyzer.** `SKIPPED` for Tagalog: no maintained Philippine-language
morphological analyzer is available off the shelf, and the design doc anticipated this.

Everything about the Tagalog inventory below is a *typological* commitment, not a
hyperparameter, so each entry carries the reason it is there.

In [ ]:
WB = "\u2581"          # word-boundary marker, shared by every condition so decode is comparable
BINDER = "\u241f"      # classifier-span binder; asserted absent from all corpus text by G6
SPECIALS = ["<pad>", "<unk>", "<bos>", "<eos>", "<mask>"]
PAD, UNK, BOS, EOS, MASK = range(5)

VOWELS = set("aeiouAEIOU")
MAX_ONSET = 2          # native Tagalog onsets are C or CC (loan clusters: gr-, tr-, pl-)
MIN_ROOT_LEN = 3
MIN_ROOT_FREQ = 2      # rung-1 evidence floor: a root must occur standalone at least twice

# The two productive Tagalog infixes. Both are inserted after the root-initial onset, so the
# root is DISCONTINUOUS in the surface string -- the exact configuration a BPE merge operation
# cannot represent, because a merge only ever joins adjacent symbols.
TAGALOG_INFIXES: Dict[str, str] = {
    "um": "AV.BEG",    # actor voice / begun aspect:  s-um-ulat 'wrote (actor focus)'
    "in": "PV.PFV",    # patient voice / perfective:  s-in-ulat 'was written'
}

# Concatenative affixes. Both tokenizers COULD in principle discover these from frequency;
# MorphAustro-Tok merely guarantees them. They are included so that the infix contrast is not
# confounded with a general "more morphology" advantage.
TAGALOG_PREFIXES = ["ipinag", "ipag", "mang", "nang", "maka", "naka", "mag", "nag",
                    "pag", "ma", "na", "pa", "ka", "i"]
TAGALOG_SUFFIXES = ["han", "hin", "an", "in"]

# Indonesian/Malay numeral-classifier construction: NUM CLF N (tiga ekor kucing).
MALAY_NUMERALS = {"satu", "dua", "tiga", "empat", "lima", "enam", "tujuh", "delapan",
                  "lapan", "sembilan", "sepuluh", "beberapa", "banyak", "berapa"}
MALAY_CLASSIFIERS = {"ekor": "animate non-human", "orang": "human", "buah": "generic/inanimate",
                     "batang": "long rigid", "biji": "small round", "helai": "thin flat",
                     "bidang": "expanse", "butir": "granular", "keping": "slab",
                     "potong": "cut piece", "pucuk": "letters/firearms", "kuntum": "flowers",
                     "tangkai": "stalked", "pasang": "pairs", "lembar": "sheets",
                     "bilah": "bladed", "utas": "cordlike", "carik": "torn strip"}

# HELD-OUT TEST FIXTURE for the infix-segmentation correctness gate (G1). This is a gate
# fixture, not a resource: it is never used to build any vocabulary, and the roots it names
# must be recovered from corpus evidence, not from this list.
INFIX_GATE_FIXTURE: List[Tuple[str, str, str, int]] = [
    ("sumulat", "sulat", "um", 1), ("kumain", "kain", "um", 1),
    ("tumakbo", "takbo", "um", 1), ("bumili", "bili", "um", 1),
    ("lumakad", "lakad", "um", 1), ("humingi", "hingi", "um", 1),
    ("sinulat", "sulat", "in", 1), ("binasa", "basa", "in", 1),
    ("ginawa", "gawa", "in", 1), ("kinain", "kain", "in", 1),
    ("pinuntahan", "puntahan", "in", 1), ("tinapos", "tapos", "in", 1),
]
GATE_FIXTURE_ROOTS = {r for _, r, _, _ in INFIX_GATE_FIXTURE}


def infix_token(infix: str, onset: int) -> str:
    """Positional infix token. The offset is carried in the token itself, which is what makes
    the segmentation invertible: <INF:um@1> + 'sulat' -> insert 'um' at offset 1 -> 'sumulat'."""
    return f"<INF:{infix}@{onset}>"


def all_infix_tokens() -> List[str]:
    """Every infix token the real analyzer can emit, reserved as atomic vocabulary entries."""
    return [infix_token(i, o) for i in TAGALOG_INFIXES for o in range(1, MAX_ONSET + 1)]


def induce_root_evidence(lines: Iterable[str], min_freq: int = MIN_ROOT_FREQ) -> Set[str]:
    """Rung 1. Standalone word types above the frequency floor, from TRAINING TEXT ONLY."""
    c = Counter(w.lower().strip(".,;:!?\"'()") for ln in lines for w in ln.split())
    return {w for w, n in c.items() if n >= min_freq and w.isalpha() and len(w) >= MIN_ROOT_LEN}


def load_root_lexicon() -> Set[str]:
    """Rung 2. Empty set if the resource is absent — the ladder degrades, it does not fake."""
    if not CAPABILITIES.get("tagalog_root_lexicon", False):
        return set()
    return {ln.strip().lower() for ln in
            TAGALOG_LEXICON_PATH.read_text(encoding="utf-8").splitlines() if ln.strip()}

## §5. `MorphAustro-Tok` — the core algorithm

Two stages, and the first is the whole point.

**Stage 1 — morphological pre-segmentation.** A surface word is mapped to a sequence of
*morphs*. For an infixed form the root is emitted as a **contiguous** unit followed by a
positional infix token, so `sumulat -> ["sulat", "<INF:um@1>"]`. The root is now contiguous
in the token stream even though it is discontinuous in the string, which is precisely what no
sequence of BPE merges can achieve. Root-then-infix order (rather than infix-first) keeps the
word-boundary marker on the root, which is the head, and makes voice a discrete decision the
decoder takes *after* committing to a root.

For Indonesian/Malay, a `NUM CLF N` sequence binds `CLF` and `N` into a single span morph, so
classifier-noun compatibility is one unit rather than a fragmented one.

**Stage 2 — constrained BPE over morphs.** Merges are learned by frequency *within* morphs and
can never cross a morph boundary, because the BPE trainer only ever sees morphs as separate
pre-tokenized units. Granularity therefore stays learned and corpus-driven; only *where the
cuts may fall* is constrained.

**Invertibility is a hard requirement,** not a nicety: MT decoding must reconstruct surface
text exactly. Gate G2 asserts lossless roundtrip over 100% of corpus lines.

In [ ]:
@dataclass
class Morph:
    """One morphological unit. `covers` lists the whitespace-word indices this morph spans —
    a list, not an int, because a classifier span legitimately covers two words. Downstream
    word-level label pooling reads `covers`; nothing is allowed to assume one-piece-one-word."""
    text: str
    kind: str                                  # root|infix|prefix|suffix|classifier_span|plain
    covers: List[int] = field(default_factory=list)
    meta: Dict[str, Any] = field(default_factory=dict)


class TagalogInfixAnalyzer:
    """Supervision-ladder rungs 1-2 for `-um-` / `-in-` infixation."""

    def __init__(self, root_evidence: Set[str], lexicon: Optional[Set[str]] = None,
                 rung: int = 2) -> None:
        self.root_evidence = root_evidence
        self.lexicon = lexicon or set()
        self.rung = rung
        self.stats: Counter = Counter()

    def _attested(self, root: str) -> bool:
        if self.rung >= 2 and root in self.lexicon:
            self.stats["by_lexicon"] += 1
            return True
        if root in self.root_evidence:
            self.stats["by_corpus"] += 1
            return True
        return False

    def analyze(self, word: str) -> Optional[Tuple[str, str, int]]:
        """-> (lowercased root, infix, onset) or None. Shortest onset wins, so `s-um-ulat`
        is preferred over any longer-onset reading."""
        w = word.lower()
        if not w.isalpha():
            return None
        for onset in range(1, MAX_ONSET + 1):
            # The onset must be consonantal: a vowel-initial root takes um- as a PREFIX
            # (alis -> umalis), which is the concatenative stripper's job, not this one.
            if any(c in VOWELS for c in w[:onset]):
                continue
            for inf in TAGALOG_INFIXES:
                k = len(inf)
                if len(w) < onset + k + MIN_ROOT_LEN or w[onset:onset + k] != inf:
                    continue
                root = w[:onset] + w[onset + k:]
                if len(root) >= MIN_ROOT_LEN and self._attested(root):
                    self.stats[(inf, onset)] += 1
                    return root, inf, onset
        return None

    @staticmethod
    def surface_root(word: str, infix: str, onset: int) -> str:
        """Case-preserving root: excise the infix from the ORIGINAL surface form so that
        reassembly is exact for capitalised tokens ('Sumulat' -> 'Sulat' + <INF:um@1>)."""
        return word[:onset] + word[onset + len(infix):]


class ConcatenativeStripper:
    """Prefix/suffix stripping, gated on the same root evidence. Longest total affixation wins."""

    def __init__(self, root_evidence: Set[str], prefixes: Sequence[str],
                 suffixes: Sequence[str]) -> None:
        self.root_evidence = root_evidence
        self.prefixes = sorted(prefixes, key=len, reverse=True)
        self.suffixes = sorted(suffixes, key=len, reverse=True)

    def strip(self, word: str) -> Optional[Tuple[str, str, str]]:
        w = word.lower()
        best: Optional[Tuple[int, int, int]] = None
        for p in [""] + list(self.prefixes):
            if p and not w.startswith(p):
                continue
            rest = w[len(p):]
            for s in [""] + list(self.suffixes):
                if s and not rest.endswith(s):
                    continue
                root = rest[:len(rest) - len(s)] if s else rest
                if len(root) < MIN_ROOT_LEN or not (p or s):
                    continue
                if root in self.root_evidence:
                    score = len(p) + len(s)
                    if best is None or score > best[0]:
                        best = (score, len(p), len(s))
        if best is None:
            return None
        _, lp, ls = best
        return word[:lp], word[lp:len(word) - ls] if ls else word[lp:], word[len(word) - ls:] if ls else ""


class MalayClassifierBinder:
    """Detect `NUM CLF N` and bind CLF+N. Only fires with an overt numeral, because a bare
    `ekor` is a noun ('tail') — binding it unconditionally would corrupt ordinary text."""

    def __init__(self, numerals: Set[str] = None, classifiers: Set[str] = None) -> None:
        self.numerals = numerals or MALAY_NUMERALS
        self.classifiers = set(classifiers or MALAY_CLASSIFIERS)
        self.fired = 0

    @staticmethod
    def _norm(w: str) -> str:
        return w.lower().strip(".,;:!?\"'()")

    def spans(self, words: Sequence[str]) -> Dict[int, int]:
        """-> {classifier_index: noun_index}. Non-overlapping, left-to-right."""
        out: Dict[int, int] = {}
        taken: Set[int] = set()
        for i in range(len(words) - 2):
            if i in taken or i + 1 in taken or i + 2 in taken:
                continue
            num, clf = self._norm(words[i]), self._norm(words[i + 1])
            if (num in self.numerals or num.isdigit()) and clf in self.classifiers:
                out[i + 1] = i + 2
                taken.update({i + 1, i + 2})
                self.fired += 1
        return out


class MorphSegmenter:
    """Stage 1. Set `scramble_seed` to build the scrambled-morpheme control instead."""

    def __init__(self, language: str, infixer: Optional[TagalogInfixAnalyzer] = None,
                 stripper: Optional[ConcatenativeStripper] = None,
                 binder: Optional[MalayClassifierBinder] = None,
                 scramble_seed: Optional[int] = None) -> None:
        self.language = language
        self.infixer, self.stripper, self.binder = infixer, stripper, binder
        self.scramble_seed = scramble_seed
        self._rng = random.Random(scramble_seed) if scramble_seed is not None else None
        self._scramble_cache: Dict[str, Tuple[str, int]] = {}
        self.counters: Counter = Counter()

    @property
    def scrambled(self) -> bool:
        return self.scramble_seed is not None

    # ---- scrambled-morpheme control -----------------------------------------------------
    def _scramble_infix(self, word: str, true_inf: str, true_onset: int) -> Tuple[str, int]:
        """Fire on EXACTLY the word types the real analyzer fired on, always emitting exactly
        two morphs, but excise a pseudo-infix at an (offset, length) guaranteed to differ from
        the true analysis. The residue is not a real root, so paradigm members stop collapsing
        onto a shared unit — while granularity and invertibility are preserved bit for bit.
        Cached per type, so the assignment is deterministic in the seed and stable corpus-wide."""
        if word in self._scramble_cache:
            return self._scramble_cache[word]
        opts = [(o, k) for o in range(1, 4) for k in (1, 2, 3)
                if len(word) >= o + k + MIN_ROOT_LEN
                and not (k == len(true_inf) and o == true_onset)]
        if not opts:                                    # too short to derange: leave as-is
            self._scramble_cache[word] = (true_inf, true_onset)
        else:
            o, k = self._rng.choice(opts)
            self._scramble_cache[word] = (word[o:o + k], o)
        return self._scramble_cache[word]

    def _scramble_spans(self, words: Sequence[str], spans: Dict[int, int]) -> Dict[int, int]:
        """Preserve the NUMBER of bound spans and their adjacency shape (p, p+1), but move each
        to a position that is not a true classifier position. Word order and invertibility are
        untouched; only the classifier-noun selection restriction is destroyed."""
        cand = [p for p in range(len(words) - 1) if p not in spans]
        self._rng.shuffle(cand)
        chosen: List[int] = []
        used: Set[int] = set()
        for p in cand:
            if len(chosen) >= len(spans):
                break
            if p in used or p + 1 in used:
                continue
            chosen.append(p)
            used.update({p, p + 1})
        if len(chosen) < len(spans):                    # short sentence: keep true positions
            return spans
        return {p: p + 1 for p in chosen}

    # ---- segmentation -------------------------------------------------------------------
    def segment(self, words: Sequence[str]) -> List[Morph]:
        spans: Dict[int, int] = {}
        if self.binder is not None:
            spans = self.binder.spans(words)
            if self.scrambled and spans:
                spans = self._scramble_spans(words, spans)
        morphs: List[Morph] = []
        consumed: Set[int] = set(spans.values())
        for i, w in enumerate(words):
            if i in consumed:
                continue
            if i in spans:
                j = spans[i]
                morphs.append(Morph(w + BINDER + words[j], "classifier_span", [i, j],
                                    {"classifier": self.binder._norm(w)}))
                self.counters["classifier_span"] += 1
                continue
            morphs.extend(self._segment_word(w, i))
        return morphs

    def _segment_word(self, word: str, idx: int) -> List[Morph]:
        if self.infixer is not None:
            an = self.infixer.analyze(word)
            if an is not None:
                _, inf, onset = an
                if self.scrambled:
                    inf, onset = self._scramble_infix(word.lower(), inf, onset)
                self.counters["infix"] += 1
                return [Morph(TagalogInfixAnalyzer.surface_root(word, inf, onset), "root", [idx]),
                        Morph(infix_token(inf, onset), "infix", [idx],
                              {"infix": inf, "onset": onset})]
        if self.stripper is not None:
            st = self.stripper.strip(word)
            if st is not None:
                p, root, s = st
                self.counters["concat"] += 1
                out = [Morph(p, "prefix", [idx])] if p else []
                out.append(Morph(root, "root", [idx]))
                if s:
                    out.append(Morph(s, "suffix", [idx]))
                return out
        self.counters["plain"] += 1
        return [Morph(word, "plain", [idx])]


def reassemble(morphs: Sequence[Morph]) -> List[str]:
    """Inverse of `MorphSegmenter.segment`. Gate G2 asserts this is exact over the corpus.

    Reconstruction keys on `covers`, not on morph adjacency: prefix/root/suffix concatenate
    into the word they cover, an infix is re-inserted at its recorded offset, and a classifier
    span splits back into its two words. Sorting by word index restores the original order.
    """
    words: Dict[int, str] = {}
    for m in morphs:
        if m.kind == "classifier_span":
            i, j = m.covers
            a, b = m.text.split(BINDER)
            words[i], words[j] = a, b
            continue
        i = m.covers[0]
        cur = words.get(i, "")
        if m.kind == "infix":
            o, inf = m.meta["onset"], m.meta["infix"]
            words[i] = cur[:o] + inf + cur[o:]
        else:                                            # root | plain | prefix | suffix
            words[i] = cur + m.text
    return [words[i] for i in sorted(words)]

### §5.1 Stage 2 — the three tokenizer conditions

All three share one interface, one word-boundary convention, one normalisation, and one
training corpus. Only the pre-tokenization differs. Everything downstream talks to
`BaseTokenizer` and never to a concrete class, so no task can accidentally depend on which
condition it is running.

In [ ]:
from tokenizers import Tokenizer as HFTokenizer  # noqa: E402
from tokenizers import decoders, models, normalizers, pre_tokenizers, trainers  # noqa: E402


class BaseTokenizer:
    """Shared interface for every condition.

    `encode_words` returns `(ids, covers)` where `covers[k]` is the LIST of whitespace-word
    indices piece k belongs to. A list rather than an int, because a classifier span covers
    two words — and because first-subtoken labelling, which assumes one piece maps to one
    word, is the defect that reversed tokenizer-comparison signs in the Afro-Asiatic study.
    Word-level pooling reads this field; nothing else is permitted to infer alignment.
    """
    name = "base"
    condition = "BASE"

    @property
    def vocab_size(self) -> int:
        raise NotImplementedError

    def id_to_piece(self, i: int) -> str:
        raise NotImplementedError

    def encode_words(self, words: Sequence[str],
                     max_len: Optional[int] = None) -> Tuple[List[int], List[List[int]]]:
        raise NotImplementedError

    def encode(self, text: str, max_len: Optional[int] = None) -> List[int]:
        return self.encode_words(text.split(), max_len=max_len)[0]

    def decode(self, ids: Sequence[int]) -> str:
        raise NotImplementedError

    def fertility(self, lines: Sequence[str], unit: str = "char") -> float:
        """Segmentation granularity.

        `unit="char"` -> tokens per character, which is what H3 is stated in and what the
        matched sweep holds constant. `unit="word"` -> tokens per whitespace word, the
        conventional reporting quantity. The two are not interchangeable here: classifier-span
        binding merges two words into one unit, so tokens/word can fall below 1.0 for the
        Indonesian condition as a pure binding artifact, while tokens/char cannot.
        """
        n_tok = n_word = n_char = 0
        for ln in lines:
            w = ln.split()
            if not w:
                continue
            n_tok += len(self.encode_words(w)[0])
            n_word += len(w)
            n_char += sum(len(x) for x in w)
        if unit == "char":
            return n_tok / max(n_char, 1)
        if unit == "word":
            return n_tok / max(n_word, 1)
        raise ValueError(f"unit must be 'char' or 'word', got {unit!r}")

    def fertility_report(self, lines: Sequence[str]) -> Dict[str, float]:
        """Both granularity measures, so every run logs the one it matched on AND the one the
        literature reports."""
        return {"tokens_per_char": self.fertility(lines, "char"),
                "tokens_per_word": self.fertility(lines, "word")}


class MatchedBPE(BaseTokenizer):
    """Baseline. Trained on the SAME corpus lines, to the SAME vocabulary size, with the SAME
    Metaspace word-boundary convention and NFC normalisation as MorphAustro-Tok. Not an
    off-the-shelf tokenizer: whether a merge respects a morpheme boundary is decided by corpus
    frequency alone, which is exactly the failure mode under test."""
    name = "BPE"
    condition = "BPE"

    def __init__(self, corpus_lines: Sequence[str], vocab_size: int) -> None:
        tok = HFTokenizer(models.BPE(unk_token="<unk>"))
        tok.normalizer = normalizers.NFC()
        tok.pre_tokenizer = pre_tokenizers.Metaspace(replacement=WB, prepend_scheme="always")
        tok.decoder = decoders.Metaspace(replacement=WB, prepend_scheme="always")
        tok.train_from_iterator(
            corpus_lines,
            trainer=trainers.BpeTrainer(vocab_size=vocab_size, special_tokens=SPECIALS,
                                        show_progress=False, min_frequency=1))
        self._tk = tok
        self._itos = [""] * tok.get_vocab_size()
        for p, i in tok.get_vocab().items():
            self._itos[i] = p

    @property
    def vocab_size(self) -> int:
        return self._tk.get_vocab_size()

    def id_to_piece(self, i: int) -> str:
        return self._itos[i] if 0 <= i < len(self._itos) else ""

    def encode_words(self, words, max_len=None):
        enc = self._tk.encode(list(words), is_pretokenized=True)
        ids = list(enc.ids)
        covers = [[w if w is not None else 0] for w in enc.word_ids]
        if max_len is not None:
            ids, covers = ids[:max_len], covers[:max_len]
        return ids, covers

    def decode(self, ids):
        ids = [i for i in ids if i not in (PAD, BOS, EOS, MASK)]
        return unicodedata.normalize("NFC", self._tk.decode(ids)).strip()


class MorphAustroTok(BaseTokenizer):
    """MorphAustro-Tok. Stage 1 morph pre-segmentation, then BPE constrained to morph interiors.

    The constraint is enforced structurally rather than by a merge-time veto: the BPE trainer
    is fed morphs as already-pre-tokenized units, so no merge it can learn spans a morph
    boundary. Infix tokens are registered as atomic special tokens, so `<INF:um@1>` is one id
    and is never fragmented.
    """
    name = "MorphAustro-Tok"

    def __init__(self, corpus_lines: Sequence[str], vocab_size: int,
                 segmenter: MorphSegmenter, condition: str = "MORPHAUSTRO") -> None:
        self.seg = segmenter
        self.condition = condition
        # The real analyzer's infix tokens are enumerable in advance and reserved as atomic
        # vocabulary entries so <INF:um@1> is never fragmented by a merge.
        infix_specials = all_infix_tokens()
        self._infix_specials = infix_specials
        morph_lines: List[List[str]] = []
        for ln in corpus_lines:
            ws = ln.split()
            if not ws:
                continue
            morph_lines.append(self._morph_strings(self.seg.segment(ws)))
        tok = HFTokenizer(models.BPE(unk_token="<unk>"))
        tok.normalizer = normalizers.NFC()
        # Whitespace pre-tokenization over already-morph-separated text: the WB marker is
        # attached by _morph_strings to the FIRST morph of each word, so the boundary
        # convention matches MatchedBPE piece for piece.
        tok.pre_tokenizer = pre_tokenizers.WhitespaceSplit()
        tok.train_from_iterator(
            (" ".join(m) for m in morph_lines),
            trainer=trainers.BpeTrainer(
                vocab_size=vocab_size,
                special_tokens=SPECIALS + infix_specials,
                show_progress=False, min_frequency=1))
        # Scrambled pseudo-infix tokens are discovered during segmentation, not enumerable in
        # advance; add them after training so they too are atomic and the id space is complete.
        discovered = sorted({m for line in morph_lines for m in line if m.startswith("<INF:")}
                            - set(infix_specials))
        if discovered:
            tok.add_special_tokens(discovered)
        self._tk = tok
        self._itos = [""] * tok.get_vocab_size()
        for p, i in tok.get_vocab().items():
            self._itos[i] = p

    @staticmethod
    def _morph_strings(morphs: Sequence[Morph]) -> List[str]:
        """Morph surface strings with WB on the first morph of each word. Infix tokens keep
        their angle-bracket form and never receive WB, since they are not word-initial."""
        out: List[str] = []
        seen: Set[int] = set()
        for m in morphs:
            if m.kind == "infix":
                out.append(m.text)
                continue
            first = m.covers[0] not in seen
            seen.add(m.covers[0])
            out.append((WB if first else "") + m.text)
        return out

    @property
    def vocab_size(self) -> int:
        return self._tk.get_vocab_size()

    def id_to_piece(self, i: int) -> str:
        return self._itos[i] if 0 <= i < len(self._itos) else ""

    def encode_words(self, words, max_len=None):
        morphs = self.seg.segment(list(words))
        strings = self._morph_strings(morphs)
        ids: List[int] = []
        covers: List[List[int]] = []
        for m, s in zip(morphs, strings):
            enc = self._tk.encode(s, add_special_tokens=False)
            pieces = enc.ids or [UNK]
            ids.extend(pieces)
            covers.extend([list(m.covers)] * len(pieces))
        if max_len is not None:
            ids, covers = ids[:max_len], covers[:max_len]
        return ids, covers

    def decode(self, ids):
        """Rebuild morphs from pieces, then invert Stage 1. Infix tokens are applied to the
        word currently under construction; the binder splits a classifier span back into two."""
        pieces = [self.id_to_piece(i) for i in ids if i not in (PAD, BOS, EOS, MASK)]
        words: List[str] = []
        cur = ""
        for p in pieces:
            if p.startswith("<INF:") and p.endswith(">"):
                body = p[len("<INF:"):-1]
                inf, o = body.rsplit("@", 1)
                try:
                    o = int(o)
                except ValueError:
                    continue
                o = min(o, len(cur))
                cur = cur[:o] + inf + cur[o:]
            elif p.startswith(WB):
                if cur:
                    words.append(cur)
                cur = p[len(WB):]
            else:
                cur += p
        if cur:
            words.append(cur)
        text = " ".join(words).replace(BINDER, " ")
        return unicodedata.normalize("NFC", text).strip()


def build_condition(language: str, corpus_lines: Sequence[str], vocab_size: int,
                    condition: str, seed: int) -> BaseTokenizer:
    """Build one tokenizer condition. Built ONCE per (language, condition, fertility level)
    and reused for MT, NER and the mechanism probe, so that task rows share a config hash and
    the task-transfer comparison is well posed."""
    if condition == "BPE":
        return MatchedBPE(corpus_lines, vocab_size)

    evidence = induce_root_evidence(corpus_lines)
    lexicon = load_root_lexicon()
    scramble = seed if condition == "SCRAMBLED" else None
    if language == "tgl":
        seg = MorphSegmenter(
            "tgl",
            infixer=TagalogInfixAnalyzer(evidence, lexicon,
                                         rung=2 if lexicon else 1),
            stripper=ConcatenativeStripper(evidence, TAGALOG_PREFIXES, TAGALOG_SUFFIXES),
            scramble_seed=scramble)
    elif language in ("ind", "msa"):
        seg = MorphSegmenter(language, binder=MalayClassifierBinder(), scramble_seed=scramble)
    else:
        raise ValueError(f"no typological configuration declared for language '{language}'")
    return MorphAustroTok(corpus_lines, vocab_size, seg, condition=condition)

## §6. Fertility-matched sweep

A typology-aware tokenizer changes *how fine* and *where* it cuts at the same time. Only a
fertility-matched comparison separates them, and H3 is falsified if the MorphAustro-Tok gain
disappears once tokens-per-word is held equal.

Granularity is measured in **tokens per character**, which is how H3 is stated. The
distinction from tokens-per-word matters: classifier-span binding merges two whitespace words
into one unit, so tokens/word drops below 1.0 for the Indonesian condition as a pure artifact
of binding. Both measures are logged; only tokens/char is matched on.

`CONFIG["fertility_targets"]` is **provisional**. The levels actually used are derived in
Part 2 by `derive_fertility_targets` from the measured intersection of every condition's
achievable envelope, because each condition saturates at a different granularity and an
a-priori target can be reachable for one arm and unreachable for another — in which case the
"matched" sweep is matched for one arm only and H3 is tested against an uncontrolled contrast.

Matching is by binary search on vocabulary size per condition per target. **This breaks
vocabulary parity**, which breaks parameter parity through the embedding table. Both
comparisons are therefore run and reported separately, exactly as the Niger-Congo study
separated matched-vocabulary from matched-output-space claims:

* **matched-vocabulary** — identical `vocab_size`, fertility free to differ (the primary contrast);
* **matched-fertility** — identical tokens/word, `vocab_size` and parameter count free to
  differ, with the delta measured and logged rather than assumed negligible.

In [ ]:
def fertility_envelope(builder: Callable[[int], BaseTokenizer], probe_lines: Sequence[str],
                       lo: int, hi: int, unit: str = "char") -> Tuple[float, float]:
    """Measured achievable granularity band for one condition, (finest, coarsest).

    Targets must be chosen against this, not guessed. A morph-constrained tokenizer on a small
    corpus can saturate long before the requested vocabulary is reached, at which point
    fertility stops responding to vocabulary size and a target outside the band is simply
    unreachable — which is a fact about the corpus, not a search failure.
    """
    f_hi = builder(hi).fertility(probe_lines, unit)      # large vocab -> fewest tokens
    f_lo = builder(lo).fertility(probe_lines, unit)      # small vocab -> most tokens
    return f_hi, f_lo


def derive_fertility_targets(builders: Dict[str, Callable[[int], BaseTokenizer]],
                             probe_lines: Sequence[str], lo: int, hi: int,
                             n_levels: int = 3, unit: str = "char") -> Dict[str, Any]:
    """Choose the sweep's fertility levels from the MEASURED intersection of every condition's
    achievable envelope, rather than fixing them a priori.

    This is not a convenience. Each condition saturates at a different granularity, so an
    a-priori target can be reachable for BPE and unreachable for MorphAustro-Tok — in which
    case the "matched" sweep is matched for one arm only, and H3 is tested against a
    comparison that was never controlled. Targets are placed at the interior of the
    intersection, and an empty intersection is a hard stop.
    """
    envs = {n: fertility_envelope(mk, probe_lines, lo, hi, unit) for n, mk in builders.items()}
    finest = max(e[0] for e in envs.values())
    coarsest = min(e[1] for e in envs.values())
    if finest > coarsest:
        raise GateFailure(
            f"achievable {unit} fertility envelopes do not intersect: {envs}. No matched "
            f"sweep exists at this corpus size and vocabulary range; H3 is untestable until "
            f"one of them changes.")
    span = coarsest - finest
    inner = [finest + span * (i + 1) / (n_levels + 1) for i in range(n_levels)]
    return {"envelopes": {k: (round(a, 5), round(b, 5)) for k, (a, b) in envs.items()},
            "intersection": (round(finest, 5), round(coarsest, 5)),
            "targets": [round(t, 5) for t in inner], "unit": unit}


def match_fertility(builder: Callable[[int], BaseTokenizer], probe_lines: Sequence[str],
                    target: float, lo: int, hi: int, tol: float, unit: str = "char",
                    max_iter: int = 12) -> Tuple[BaseTokenizer, int, float]:
    """Binary-search vocabulary size for a target granularity.

    Fertility is monotone decreasing in vocabulary size (a larger vocabulary means more merges
    and so fewer pieces), which is what makes bisection valid. Non-monotone excursions from
    tie-breaking in the merge table are absorbed by the tolerance band.

    RAISES if the target lies outside the measured envelope, rather than returning a
    best-effort tokenizer that would make G7 fail later with a less diagnosable message.
    """
    f_min, f_max = fertility_envelope(builder, probe_lines, lo, hi, unit)
    if not (f_min - tol <= target <= f_max + tol):
        raise GateFailure(
            f"fertility target {target} ({unit}) is outside the achievable envelope "
            f"[{f_min:.4f}, {f_max:.4f}] for vocab in [{lo}, {hi}]. Re-choose "
            f"CONFIG['fertility_targets'] against the measured envelope, or widen the "
            f"vocabulary search range — do not widen the tolerance.")
    best: Optional[Tuple[BaseTokenizer, int, float]] = None
    for _ in range(max_iter):
        mid = (lo + hi) // 2
        tk = builder(mid)
        f = tk.fertility(probe_lines, unit)
        if best is None or abs(f - target) < abs(best[2] - target):
            best = (tk, mid, f)
        if abs(f - target) <= tol:
            return tk, mid, f
        if f > target:                       # too many tokens -> enlarge the vocabulary
            lo = mid + 1
        else:
            hi = mid - 1
        if lo > hi:
            break
    assert best is not None
    return best

## §7. Validation gates

Every gate is **runnable code that raises**. A failed gate aborts the notebook; it does not
print a warning and continue. The design doc's rationale is that a mechanism which is not
demonstrably active makes every downstream number uninterpretable, so the gates run *before*
any hypothesis is reported.

| Gate | Asserts | Defect it guards against |
|---|---|---|
| G1 | held-out infixed forms segment as root+infix | the mechanism is inert; everything downstream void |
| G2 | lossless roundtrip on 100% of corpus lines | MT decoding cannot reconstruct surface text |
| G3 | vocabulary-count parity across conditions | illusory parameter matching (caught twice before) |
| G4 | scrambled control has identical morph granularity | the control confounds granularity with typology |
| G5 | scrambled assignment is a genuine derangement | the "control" silently reproduces the real analysis |
| G6 | marker codepoints absent from corpus text | binder/WB collision silently corrupts text |
| G7 | achieved fertility within tolerance of target | H3's matched sweep is not actually matched |
| G8 | inference-path parameter parity across conditions | capacity difference explains the result |

G9 (test-set contamination), G10 (decoder future-leakage) and G11 (first-subtoken labelling
audit) attach to components defined in Parts 2-3 and are registered here as pending so their
absence is visible rather than forgotten.

In [ ]:
def _gate(name: str, passed: bool, observed: Any, expected: Any = None,
          context: Optional[Dict[str, Any]] = None) -> None:
    LOG.gate(name, passed, observed, expected, context)
    if not passed:
        raise GateFailure(f"{name}: observed {observed!r}, expected {expected!r}")


PENDING_GATES = {
    "G9_test_set_contamination": "Part 2 — FLORES/WikiANN splits vs pretraining-corpus overlap",
    "G10_decoder_future_leakage": "Part 3 — causality unit test on MT decoding order",
    "G11_first_subtoken_audit": "Part 3 — assert word-level pooling, never first-subtoken",
}


def gate_G1_infix_segmentation(segmenter: MorphSegmenter,
                               fixture=INFIX_GATE_FIXTURE) -> None:
    """G1 — the mechanism-active check. Held-out infixed forms must segment as root+infix, with
    the root recovered from corpus evidence rather than from the fixture list."""
    failures = []
    for surface, want_root, want_inf, want_onset in fixture:
        morphs = segmenter.segment([surface])
        got = [(m.text, m.kind) for m in morphs]
        ok = (len(morphs) == 2 and morphs[0].kind == "root" and morphs[1].kind == "infix"
              and morphs[0].text.lower() == want_root
              and morphs[1].meta.get("infix") == want_inf
              and morphs[1].meta.get("onset") == want_onset)
        if not ok:
            failures.append({"surface": surface, "expected": (want_root, want_inf, want_onset),
                             "got": got})
    _gate("G1_infix_segmentation", not failures,
          f"{len(fixture) - len(failures)}/{len(fixture)} held-out infixed forms segmented "
          f"as root+infix", "12/12", {"failures": failures})


def gate_G2_roundtrip(tok: BaseTokenizer, lines: Sequence[str], sample: int = 2000) -> None:
    """G2 — invertibility. Segmentation that cannot be undone makes MT decoding meaningless."""
    rng = random.Random(GLOBAL_SEED)
    probe = lines if len(lines) <= sample else rng.sample(list(lines), sample)
    bad = []
    for ln in probe:
        words = ln.split()
        if not words:
            continue
        want = unicodedata.normalize("NFC", " ".join(words))
        got = tok.decode(tok.encode_words(words)[0])
        if got != want:
            bad.append({"want": want[:120], "got": got[:120]})
    rate = 1.0 - len(bad) / max(len(probe), 1)
    _gate(f"G2_roundtrip[{tok.condition}]", not bad, f"{rate:.6f} exact over n={len(probe)}",
          "1.000000", {"examples": bad[:5]})


def gate_G3_vocab_parity(tokenizers: Dict[str, BaseTokenizer], requested_vocab: int,
                         max_spread_frac: float = 0.02) -> None:
    """G3 — output-space parity, asserted in code rather than trusted from config.

    Equal *requested* vocabulary does not imply equal *achieved* vocabulary: BPE saturates on
    a real corpus, whereas a morph-constrained inventory is finite and may not. For sequence
    labelling a shortfall is largely benign (the output space is the tag set); for generation
    the vocabulary IS the output space, so a divergence changes prediction difficulty and is a
    live confound on parameter matching through the embedding table.

    The gate gates on the SPREAD of achieved sizes as a fraction of the request. If it fires,
    the correct response is to lower `vocab_size` until every condition saturates — not to
    widen the tolerance.
    """
    achieved = {c: t.vocab_size for c, t in tokenizers.items()}
    shortfall = {c: round(1.0 - v / requested_vocab, 6) for c, v in achieved.items()}
    spread_frac = (max(achieved.values()) - min(achieved.values())) / max(requested_vocab, 1)
    _gate("G3_vocab_parity", spread_frac <= max_spread_frac,
          {"requested": requested_vocab, "achieved": achieved, "shortfall": shortfall,
           "spread_frac": round(spread_frac, 6)},
          f"spread_frac <= {max_spread_frac}",
          {"remedy": "lower CONFIG['vocab_size'] until all conditions saturate"})


def gate_G4_scramble_granularity(real: MorphSegmenter, scram: MorphSegmenter,
                                 lines: Sequence[str]) -> None:
    """G4 — the scrambled control must differ in linguistic content ONLY. If morph counts move,
    the control is confounded with granularity and cannot falsify anything."""
    diffs = []
    for ln in lines:
        w = ln.split()
        if not w:
            continue
        a, b = len(real.segment(w)), len(scram.segment(w))
        if a != b:
            diffs.append((ln[:60], a, b))
    _gate("G4_scramble_granularity", not diffs,
          f"{len(lines) - len(diffs)}/{len(lines)} lines with identical morph counts",
          "all lines identical", {"examples": diffs[:5]})


def gate_G5_scramble_derangement(real: MorphSegmenter, scram: MorphSegmenter,
                                 words: Sequence[str], min_rate: float = 0.95) -> None:
    """G5 — the control must actually be scrambled. A 'control' that reproduces the real
    analysis on most types tests nothing."""
    same = total = 0
    for w in words:
        a = real.segment([w])
        b = scram.segment([w])
        if len(a) == 2 and a[1].kind == "infix":
            total += 1
            same += int(len(b) == 2 and b[1].text == a[1].text)
    rate = 1.0 - same / max(total, 1)
    _gate("G5_scramble_derangement", total > 0 and rate >= min_rate,
          f"derangement rate {rate:.4f} over {total} infixed types", f">= {min_rate}")


def gate_G6_marker_disjointness(lines: Sequence[str]) -> None:
    """G6 — WB and BINDER must not occur in corpus text, or reassembly silently corrupts it."""
    hits = {m: sum(ln.count(m) for ln in lines) for m in (WB, BINDER, "<INF:")}
    _gate("G6_marker_disjointness", sum(hits.values()) == 0,
          {repr(k): v for k, v in hits.items()}, "all zero")


def gate_G7_fertility_match(achieved: Dict[str, float], target: float,
                            tol: float = None, unit: str = None) -> None:
    """G7 — H3 requires the sweep to be genuinely matched. An unmatched 'matched' sweep proves
    nothing, and the failure mode is silent: every condition still trains and still scores."""
    tol = CONFIG["fertility_tol"] if tol is None else tol
    unit = CONFIG["fertility_unit"] if unit is None else unit
    worst = max(abs(v - target) for v in achieved.values())
    _gate(f"G7_fertility_match@{target}({unit})", worst <= tol,
          {k: round(v, 5) for k, v in achieved.items()},
          f"|f - {target}| <= {tol} tokens per {unit}")


def gate_G8_parameter_parity(param_counts: Dict[str, int], tol_frac: float = 0.005) -> None:
    """G8 — inference-path parameter parity. Training-only heads are excluded and reported
    separately; folding them in is how a capacity difference gets mistaken for an effect."""
    lo, hi = min(param_counts.values()), max(param_counts.values())
    delta = (hi - lo) / max(lo, 1)
    _gate("G8_parameter_parity", delta <= tol_frac,
          {k: v for k, v in param_counts.items()} | {"relative_delta": round(delta, 6)},
          f"relative delta <= {tol_frac}")


for _g, _why in PENDING_GATES.items():
    print(f"  [PENDING] {_g}: {_why}")

  [PENDING] G9_test_set_contamination: Part 2 — FLORES/WikiANN splits vs pretraining-corpus overlap
  [PENDING] G10_decoder_future_leakage: Part 3 — causality unit test on MT decoding order
  [PENDING] G11_first_subtoken_audit: Part 3 — assert word-level pooling, never first-subtoken


## §8. Models

Small, from scratch, 1-9M parameters, per the Afro-Asiatic rationale: a model large enough to
memorise its way around a segmentation makes the question unanswerable. All conditions train
from random initialisation — MorphAustro-Tok induces its own vocabulary, so no pretrained
checkpoint applies and transplanting embeddings would introduce a confound larger than the
effect under study. **Absolute scores will be low; only controlled contrasts are interpretable.**

Geometry is byte-identical across conditions. The only thing that differs is the id stream.

In [ ]:
class SinusoidalPE(nn.Module):
    def __init__(self, d_model: int, max_len: int = 4096) -> None:
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(max_len, dtype=torch.float32).unsqueeze(1)
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(0), persistent=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x + self.pe[:, : x.size(1)]


class TinyEncoder(nn.Module):
    """Encoder used for NER/tagging and for both mechanism probes."""

    def __init__(self, vocab_size: int, n_labels: int, cfg: Dict[str, Any],
                 pooled: bool = False) -> None:
        super().__init__()
        d = cfg["d_model"]
        self.emb = nn.Embedding(vocab_size, d, padding_idx=PAD)
        self.pe = SinusoidalPE(d, cfg["max_len_tag"] + 8)
        layer = nn.TransformerEncoderLayer(
            d_model=d, nhead=cfg["n_heads"], dim_feedforward=cfg["d_ff"],
            dropout=cfg["dropout"], batch_first=True, norm_first=True,
            activation="gelu")
        self.enc = nn.TransformerEncoder(layer, num_layers=cfg["n_layers_enc"])
        self.drop = nn.Dropout(cfg["dropout"])
        self.head = nn.Linear(d, n_labels)
        self.pooled = pooled                    # True -> sentence-level probe, False -> tagging

    def encode(self, ids: torch.Tensor) -> torch.Tensor:
        mask = ids.eq(PAD)
        h = self.enc(self.pe(self.emb(ids)), src_key_padding_mask=mask)
        return self.drop(h)

    def forward(self, ids: torch.Tensor,
                pool_matrix: Optional[torch.Tensor] = None) -> torch.Tensor:
        """`pool_matrix` [B, W, T] maps pieces to whitespace words (see §9.2). It is REQUIRED
        for token-level tasks: this model has no first-subtoken path to fall back on, by design."""
        h = self.encode(ids)
        if self.pooled:
            m = (~ids.eq(PAD)).float().unsqueeze(-1)
            return self.head((h * m).sum(1) / m.sum(1).clamp(min=1.0))
        if pool_matrix is None:
            raise ValueError(
                "token-level forward requires pool_matrix; first-subtoken labelling is "
                "prohibited (see G11 and the Afro-Asiatic sign reversal it caused)")
        return self.head(torch.bmm(pool_matrix, h))


class TinySeq2Seq(nn.Module):
    """4+4 encoder-decoder for MT, tied output embeddings, shared source/target vocabulary."""

    def __init__(self, vocab_size: int, cfg: Dict[str, Any]) -> None:
        super().__init__()
        d = cfg["d_model"]
        self.emb = nn.Embedding(vocab_size, d, padding_idx=PAD)
        self.pe = SinusoidalPE(d, cfg["max_len_mt"] + 8)
        self.tf = nn.Transformer(
            d_model=d, nhead=cfg["n_heads"], num_encoder_layers=cfg["n_layers_enc"],
            num_decoder_layers=cfg["n_layers_dec"], dim_feedforward=cfg["d_ff"],
            dropout=cfg["dropout"], batch_first=True, norm_first=True, activation="gelu")
        self.out = nn.Linear(d, vocab_size, bias=False)
        self.out.weight = self.emb.weight                   # tied

    def forward(self, src: torch.Tensor, tgt_in: torch.Tensor) -> torch.Tensor:
        causal = nn.Transformer.generate_square_subsequent_mask(
            tgt_in.size(1), device=tgt_in.device)
        h = self.tf(self.pe(self.emb(src)), self.pe(self.emb(tgt_in)),
                    src_key_padding_mask=src.eq(PAD),
                    tgt_key_padding_mask=tgt_in.eq(PAD),
                    memory_key_padding_mask=src.eq(PAD),
                    tgt_mask=causal, tgt_is_causal=True)
        return self.out(h)


def count_parameters(model: nn.Module, inference_path_only: bool = True) -> int:
    """Parameter count on the INFERENCE path. Auxiliary heads that exist only during training
    are excluded and reported separately; folding them in is how illusory parameter matching
    happened twice in the prior studies."""
    total = 0
    for name, p in model.named_parameters():
        if inference_path_only and name.startswith("aux_"):
            continue
        total += p.numel()
    return total

## §9. Training and evaluation harness

### §9.1 Seeding, checkpoint fingerprinting, budget-aware halting

A checkpoint carries an **architectural fingerprint** and a result row carries a
**configuration fingerprint**. A mismatch invalidates the artefact rather than resuming a
different experiment under the same filename — the stale-artefact defect from the prior study.

In [ ]:
def set_seed(seed: int) -> None:
    """Per-run seeding. Called at the top of every run so that the run's stream is a function
    of its own seed and not of how many runs preceded it."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def arch_fingerprint(model: nn.Module, vocab_size: int) -> str:
    shapes = [f"{n}:{tuple(p.shape)}" for n, p in sorted(model.named_parameters())]
    return stable_hash(json.dumps({"vocab": vocab_size, "shapes": shapes}))


@dataclass
class RunSpec:
    """One cell of the run matrix. `config_hash` is shared by every task in the run, which is
    what makes the MT-vs-NER task-transfer comparison well posed."""
    language: str
    condition: str
    fertility_level: str
    target_fertility: float
    seed: int
    vocab_size: int

    @property
    def run_id(self) -> str:
        stem = f"{self.language}_{self.condition}_{self.fertility_level}_s{self.seed}"
        return ("SMOKE__" + stem) if QUICK_SMOKE else stem

    @property
    def config_hash(self) -> str:
        return config_hash(language=self.language, condition=self.condition,
                           fertility_level=self.fertility_level, seed=self.seed,
                           vocab_size=self.vocab_size,
                           target_fertility=self.target_fertility, smoke=QUICK_SMOKE)


def ckpt_path(spec: RunSpec, task: str) -> Path:
    return OUTPUT_ROOT / "checkpoints" / f"{spec.run_id}__{task}__{spec.config_hash}.pt"


def save_ckpt(model: nn.Module, opt: torch.optim.Optimizer, step: int, spec: RunSpec,
              task: str, vocab_size: int, extra: Optional[Dict[str, Any]] = None) -> Path:
    p = ckpt_path(spec, task)
    torch.save({"model": model.state_dict(), "opt": opt.state_dict(), "step": step,
                "arch_fingerprint": arch_fingerprint(model, vocab_size),
                "config_hash": spec.config_hash, "spec": asdict(spec),
                "extra": extra or {}}, p)
    return p


def load_ckpt(model: nn.Module, opt: Optional[torch.optim.Optimizer], spec: RunSpec,
              task: str, vocab_size: int) -> int:
    """Resume only if BOTH fingerprints match. Otherwise the artefact is stale and is refused."""
    p = ckpt_path(spec, task)
    if not p.exists():
        return 0
    blob = torch.load(p, map_location=DEVICE)
    if blob.get("config_hash") != spec.config_hash:
        raise GateFailure(f"stale checkpoint {p.name}: config hash mismatch")
    if blob.get("arch_fingerprint") != arch_fingerprint(model, vocab_size):
        raise GateFailure(f"stale checkpoint {p.name}: architecture fingerprint mismatch")
    model.load_state_dict(blob["model"])
    if opt is not None:
        opt.load_state_dict(blob["opt"])
    return int(blob["step"])


def make_optimizer(model: nn.Module) -> torch.optim.Optimizer:
    decay, no_decay = [], []
    for n, p in model.named_parameters():
        (no_decay if p.ndim < 2 else decay).append(p)
    return torch.optim.AdamW(
        [{"params": decay, "weight_decay": CONFIG["weight_decay"]},
         {"params": no_decay, "weight_decay": 0.0}],
        lr=CONFIG["lr"], betas=(0.9, 0.98), eps=1e-9)


def lr_at(step: int) -> float:
    """Inverse-sqrt schedule with linear warmup. Identical across every condition."""
    w = CONFIG["warmup_steps"]
    return CONFIG["lr"] * min((step + 1) / w, (w / (step + 1)) ** 0.5)

### §9.2 Word-level label pooling

**Non-negotiable.** Labels are pooled to whitespace words, never assigned to the first
subtoken. First-subtoken labelling is what reversed tokenizer-comparison signs in the
Afro-Asiatic study, and it is *structurally* wrong here: MorphAustro-Tok emits the root
before the infix, so the "first subtoken" of `sumulat` is a piece of `sulat` under one
condition and a piece of `sum` under another — the label would attach to non-comparable
material across conditions.

`build_pool_matrix` produces a `[W, T]` row-stochastic matrix from the `covers` field. A
classifier span contributes its representation to *both* words it covers, which is why
`covers` is a list.

In [ ]:
def build_pool_matrix(covers: Sequence[Sequence[int]], n_words: int,
                      n_pieces: int) -> torch.Tensor:
    """Row-stochastic [W, T] mean-pooling matrix. Row w averages every piece covering word w."""
    m = torch.zeros(n_words, n_pieces)
    for t, cov in enumerate(covers[:n_pieces]):
        for w in cov:
            if 0 <= w < n_words:
                m[w, t] = 1.0
    row = m.sum(1, keepdim=True)
    unaligned = int((row.squeeze(1) == 0).sum().item())
    if unaligned:
        # A word with no surviving piece (truncation) must be masked out downstream, never
        # silently scored: it is recorded on the batch and excluded from the loss.
        row = row.clamp(min=1.0)
    return m / row


def pad_batch(seqs: Sequence[Sequence[int]], max_len: int,
              pad: int = PAD) -> torch.Tensor:
    out = torch.full((len(seqs), max_len), pad, dtype=torch.long)
    for i, s in enumerate(seqs):
        s = s[:max_len]
        out[i, : len(s)] = torch.tensor(s, dtype=torch.long)
    return out

### §9.3 Training loops

Both loops are budget-aware: they checkpoint and exit cleanly when the per-run wall-clock
ceiling is hit, rather than overrunning the 10-hour cap. Every evaluation writes a row
through `LOG`, so a halted run still contributes its completed measurements.

In [ ]:
def train_tagging(model: TinyEncoder, batches: Callable[[int], Iterable[Dict[str, Any]]],
                  eval_fn: Callable[[TinyEncoder], Dict[str, float]], spec: RunSpec,
                  task: str, vocab_size: int, n_steps: int,
                  ignore_index: int = -100) -> Dict[str, float]:
    """Token-level training with word-level pooling. `batches(seed)` must yield dicts with
    `ids` [B,T], `pool` [B,W,T] and `labels` [B,W]."""
    set_seed(spec.seed)
    model.to(DEVICE)
    opt = make_optimizer(model)
    scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)
    step = load_ckpt(model, opt, spec, task, vocab_size)
    best: Dict[str, float] = {}
    model.train()
    stream = iter(batches(spec.seed))
    while step < n_steps:
        try:
            b = next(stream)
        except StopIteration:
            stream = iter(batches(spec.seed + step))
            continue
        for g in opt.param_groups:
            g["lr"] = lr_at(step)
        ids = b["ids"].to(DEVICE)
        pool = b["pool"].to(DEVICE)
        labels = b["labels"].to(DEVICE)
        with torch.autocast(device_type=DEVICE.type, dtype=AMP_DTYPE, enabled=USE_AMP):
            logits = model(ids, pool_matrix=pool)
            loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), labels.reshape(-1),
                                   ignore_index=ignore_index,
                                   label_smoothing=CONFIG["label_smoothing"])
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        nn.utils.clip_grad_norm_(model.parameters(), CONFIG["grad_clip"])
        scaler.step(opt)
        scaler.update()
        step += 1
        if step % CONFIG["eval_every"] == 0 or step == n_steps:
            model.eval()
            with torch.no_grad():
                metrics = eval_fn(model)
            model.train()
            best = metrics if not best or metrics.get("f1", 0) > best.get("f1", 0) else best
            for k, v in metrics.items():
                LOG.log(run_id=spec.run_id, language=spec.language,
                        tokenizer_condition=spec.condition,
                        fertility_level=spec.fertility_level, seed=spec.seed, task=task,
                        metric_name=f"{k}@step{step}", metric_value=v,
                        gpu_hours_used=BUDGET.elapsed_this_run / 3600.0,
                        config_hash=spec.config_hash)
            save_ckpt(model, opt, step, spec, task, vocab_size, {"metrics": metrics})
        if BUDGET.out_of_time():
            save_ckpt(model, opt, step, spec, task, vocab_size, {"halted": "budget"})
            print(f"  [BUDGET] {spec.run_id}/{task} halted cleanly at step {step}/{n_steps}")
            break
    return best


def train_mt(model: TinySeq2Seq, batches: Callable[[int], Iterable[Dict[str, Any]]],
             eval_fn: Callable[[TinySeq2Seq], Dict[str, float]], spec: RunSpec,
             vocab_size: int, n_steps: int) -> Dict[str, float]:
    """Teacher-forced seq2seq training. Shifted targets: `tgt_in = [BOS] + y[:-1]`,
    `tgt_out = y`. The causal mask is asserted by G10 in Part 3, not assumed here."""
    set_seed(spec.seed)
    model.to(DEVICE)
    opt = make_optimizer(model)
    scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)
    step = load_ckpt(model, opt, spec, "mt", vocab_size)
    best: Dict[str, float] = {}
    model.train()
    stream = iter(batches(spec.seed))
    while step < n_steps:
        try:
            b = next(stream)
        except StopIteration:
            stream = iter(batches(spec.seed + step))
            continue
        for g in opt.param_groups:
            g["lr"] = lr_at(step)
        src, tgt_in, tgt_out = (b["src"].to(DEVICE), b["tgt_in"].to(DEVICE),
                                b["tgt_out"].to(DEVICE))
        with torch.autocast(device_type=DEVICE.type, dtype=AMP_DTYPE, enabled=USE_AMP):
            logits = model(src, tgt_in)
            loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), tgt_out.reshape(-1),
                                   ignore_index=PAD,
                                   label_smoothing=CONFIG["label_smoothing"])
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        nn.utils.clip_grad_norm_(model.parameters(), CONFIG["grad_clip"])
        scaler.step(opt)
        scaler.update()
        step += 1
        if step % CONFIG["eval_every"] == 0 or step == n_steps:
            model.eval()
            with torch.no_grad():
                metrics = eval_fn(model)
            model.train()
            best = (metrics if not best or metrics.get("chrf", 0) > best.get("chrf", 0)
                    else best)
            for k, v in metrics.items():
                LOG.log(run_id=spec.run_id, language=spec.language,
                        tokenizer_condition=spec.condition,
                        fertility_level=spec.fertility_level, seed=spec.seed, task="mt",
                        metric_name=f"{k}@step{step}", metric_value=v,
                        gpu_hours_used=BUDGET.elapsed_this_run / 3600.0,
                        config_hash=spec.config_hash)
            save_ckpt(model, opt, step, spec, "mt", vocab_size, {"metrics": metrics})
        if BUDGET.out_of_time():
            save_ckpt(model, opt, step, spec, "mt", vocab_size, {"halted": "budget"})
            print(f"  [BUDGET] {spec.run_id}/mt halted cleanly at step {step}/{n_steps}")
            break
    return best

## STOP — end of Part 1 of 4

**Defined and validated above:** Cell 0 output root and dual CSV/JSON logging · pinned
environment and run manifest · enforced GPU-hour ledger · corpus registry with SHA-256
provenance and fail-loud fetch · capability gating (two `SKIPPED` components recorded with
reasons) · the `MorphAustro-Tok` core algorithm with its matched-BPE baseline and
scrambled-morpheme control · fertility-matched vocabulary search · gates G1-G8 as raising code
· tiny encoder and seq2seq with inference-path parameter counting · word-level pooling ·
budget-aware, fingerprinted training loops for both task families.

**Deliberately not yet written**, and tracked so nothing is lost:

| Part | Contents |
|---|---|
| 2 | FLORES/OPUS/WikiANN ingestion, Malay-Indonesian pooling measurement (§3.4), G9 contamination check, task encoders |
| 3 | Beam decoding + G10 future-leakage causality test, NER scorer + G11 first-subtoken audit, the two mechanism probes |
| 4 | The 42-run matrix executor, paired bootstrap + Holm, seed-level dispersion, LaTeX/markdown tables, Limitations log |

In [ ]:
print("Part 1 defined. Nothing has been trained. Run the gate suite in Part 2 before any "
      "hypothesis is reported.")

Part 1 defined. Nothing has been trained. Run the gate suite in Part 2 before any hypothesis is reported.
